# Gateway y release signals

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/19_diseno_api/code/04_gateway_y_release_signals.ipynb)


## 1. Setup

In [ ]:
%pip install numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 2. Routing y rate limit simplificado

In [ ]:
def enrutar(path):
    if path.startswith('/v1/chat'):
        return 'chatbot-service'
    if path.startswith('/v1/usage'):
        return 'usage-service'
    return '404'

def rate_limit(requests_per_minute, limit=5):
    return 'permitido' if requests_per_minute <= limit else '429 Too Many Requests'

print(enrutar('/v1/chat'))
print(rate_limit(7))

## 3. Simulacion de canary rollout

In [ ]:
traffic = np.array([0, 10, 25, 50, 100])
error_rate = np.array([0.0, 0.7, 0.8, 1.2, 1.1])
p95 = np.array([420, 435, 445, 510, 500])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(traffic, error_rate, marker='o', label='error %')
ax.plot(traffic, p95 / 100, marker='s', label='p95 / 100')
ax.set_xlabel('trafico hacia v2 (%)')
ax.set_title('Canary: salud de la nueva version')
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## 4. Regla de rollout

In [ ]:
for t, e, l in zip(traffic, error_rate, p95):
    decision = 'rollback' if e > 1.5 or l > 520 else 'avanzar'
    print(f'trafico={t:>3}% | error={e:.1f}% | p95={l}ms -> {decision}')